# 1.1 • Coordinates describe a place

**Learning objectives:** distinguish assigning a CRS from transforming coordinates; use longitude-first axis order; verify a round trip.

**How to learn:** predict → run → explain → change one thing. Spend 45–90 minutes here.

**Data:** the default worked example is deterministic synthetic practice data near Colorado. It is not SNOTEL, a watershed survey, a burn map, or a foundation-model output.


## Before you run

A CRS supplies units, datum, and projection. EPSG:4326 uses degrees; EPSG:26913 uses meters. Changing a CRS label does not move coordinate values. Predict the approximate magnitude of a Colorado UTM easting.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import OUT, points


## Worked example: transform explicitly


In [ ]:
from pyproj import Transformer
import geopandas as gpd
from shapely.geometry import box
stations = points(12)
source = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations.longitude, stations.latitude), crs=4326)
forward = Transformer.from_crs(4326,26913,always_xy=True)
x,y = forward.transform(stations.longitude.to_numpy(),stations.latitude.to_numpy())
projected = gpd.GeoDataFrame(stations,geometry=gpd.points_from_xy(x,y),crs=26913)
np.testing.assert_allclose(projected.geometry.x, source.to_crs(26913).geometry.x)
print(source.geometry.iloc[0], projected.geometry.iloc[0])


## Verify, then overlay

The rectangle is a teaching study area, not a county boundary. Both layers must use the same CRS before overlay.


In [ ]:
back = Transformer.from_crs(26913,4326,always_xy=True)
lon,lat = back.transform(x,y)
np.testing.assert_allclose(lon,stations.longitude,atol=1e-7,rtol=0)
np.testing.assert_allclose(lat,stations.latitude,atol=1e-7,rtol=0)
area = gpd.GeoDataFrame(geometry=[box(-105.32,39.88,-105.12,40.08)],crs=4326).to_crs(26913)
ax=area.boundary.plot(figsize=(6,6)); projected.plot(ax=ax,color='darkred')
ax.set(title='Synthetic study points in UTM 13N',xlabel='Easting (m)',ylabel='Northing (m)')
plt.show()


## Your turn

Transform the points to EPSG:3857. Compare the numeric coordinates. Then intentionally use `set_crs(26913, allow_override=True)` on the original points and explain the faulty result.

Use a new cell below to try this before opening the solution.


In [ ]:
# Write your experiment here. The worked example above is already complete.


<details><summary>Solution and reasoning — open after trying</summary>

Use `source.to_crs(3857)`. Web Mercator and UTM both use meter coordinates but have different origins and distortion. `set_crs` changes metadata only: longitude near −105 becomes an impossible UTM easting for this location. Keep the correct `source` unchanged by assigning experiments to new variables.

</details>


## Check yourself

Why can two meter-based CRSs still disagree? Why does `always_xy=True` matter? Expected: all round-trip assertions pass and all 12 points lie inside the study rectangle.

**Exit ticket:** write one result, one explanation, and one limitation. Save the notebook with your own observations.


## Transfer to real data

Use a public NRCS station CSV with longitude/latitude and a USGS WBD polygon layer. Record source URL, retrieval date and native CRS; do not guess a missing CRS. Replace the fixtures only after plotting both layers separately. See `docs/REAL_DATA.md`.
